In [ ]:
# Severe Rainfall Frequency Analysis (PRISM)

This notebook identifies extreme rainfall events in observed PRISM precipitation data and quantifies **annual event frequency**.

Extreme events are defined using a **fixed daily precipitation threshold**, applied consistently across time to support non-stationary analysis.

This notebook produces **event counts only**.  
Comparison across historical periods (“epochs”) is performed in `03_epoch_comparison.ipynb`.

---

## Purpose

- Flag extreme rainfall events using a fixed threshold
- Count extreme events by calendar year
- Generate frequency tables for downstream comparison and risk analysis

---

## Key Concepts

- **Frequency vs. intensity**  
  This analysis evaluates how often extreme events occur, not storm magnitude.

- **Consistency across time**  
  A fixed threshold is used to detect change without assuming stationarity.

- **Modularity**  
  Outputs are exported for use in later notebooks and external replication.

---

## Inputs

- Cleaned daily precipitation table from:
  - `01_prism_precip_ingest.ipynb`

---

## Outputs

- Annual extreme rainfall event counts
- Tables suitable for:
  - epoch comparison
  - mapping
  - replication in other regions

---

## Notes

- This notebook does not perform epoch-based aggregation.
- Comparison across historical periods is performed in `03_epoch_comparison.ipynb`.


In [ ]:
from pathlib import Path
import pandas as pd

from src.prism_extremes import (
    flag_extreme_events,
    count_annual_extremes
)

# Repo-aware paths
REPO_ROOT = Path("..").resolve()
OUTPUTS_DIR = REPO_ROOT / "outputs"

print("Repo root:", REPO_ROOT)


In [ ]:
# -----------------------------
# USER CONFIGURATION
# -----------------------------

# Input from Notebook 01
CLEANED_PRISM_PATH = REPO_ROOT / "outputs" / "tables" / "prism_daily_cleaned.csv"

# Column names (must match Notebook 01 output)
DATE_COL = "Date"
PRECIP_COL = "inchesPpt"

# Extreme rainfall threshold (inches)
EXTREME_THRESHOLD_IN = 3.0


In [ ]:
if not CLEANED_PRISM_PATH.exists():
    raise FileNotFoundError(
        f"Missing cleaned PRISM data at {CLEANED_PRISM_PATH}. "
        "Run 01_prism_precip_ingest.ipynb first."
    )


In [ ]:
df = pd.read_csv(
    CLEANED_PRISM_PATH,
    parse_dates=[DATE_COL]
).set_index(DATE_COL)

df.head()


In [ ]:
extreme_flags = flag_extreme_events(
    df,
    precip_col=PRECIP_COL,
    threshold_in=EXTREME_THRESHOLD_IN
)

print("Total extreme-event days:", extreme_flags.sum())


In [ ]:
annual_counts = count_annual_extremes(extreme_flags)

annual_counts.head()


In [ ]:
annual_counts_df = annual_counts.reset_index()
annual_counts_df.columns = ["year", "extreme_event_count"]

annual_counts_df.head()


In [ ]:
OUTPUT_PATH = OUTPUTS_DIR / "tables" / "prism_annual_extreme_counts.csv"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

annual_counts_df.to_csv(OUTPUT_PATH, index=False)

print(f"Annual extreme-event counts saved to {OUTPUT_PATH}")


In [ ]:
annual_counts_df.describe()


In [ ]:
## Next Steps

This notebook produces annual extreme rainfall event counts derived from observed PRISM data using a fixed threshold definition.

These annual frequency outputs are used directly in:

- `03_epoch_comparison.ipynb`  
  → to compare extreme-event frequency across historical time windows without assuming stationarity

Users may proceed directly to the next notebook after confirming this step completes successfully.